# RF-Based Drone Detection and Classification Using Deep Learning
### UC Berkeley ML/AI Professional Certificate — Capstone Project
**Author:** Aya Mundhenk  
**GitHub:** https://github.com/ayamundhenk/berkeley_capstone


---
## 1. Problem Statement

**Research Question:**  
Can radio frequency (RF) signal data alone — without cameras, radar, or GPS — be used to reliably detect the presence of a drone and classify which type it is?

### Background

The rapid proliferation of consumer and commercial unmanned aerial vehicles (UAVs) has created growing security and airspace management challenges, particularly in restricted or sensitive environments such as airports, prisons, stadiums, and military installations. Most existing counter-drone systems rely on cameras (fail at night/poor weather), radar (struggles with small/slow drones), or GPS tracking (only works if the drone cooperates). RF-based detection closes that gap: **every drone must communicate with its controller**, whether or not it wants to be tracked.

### Goals

This project builds a machine learning model that:
1. **Detects** whether a drone is present in a given RF environment (binary classification)
2. **Classifies** the specific drone type — AR, Bebop, or Phantom — and its operating mode (multi-class classification)

Performance will be evaluated across varying signal-to-noise (SNR) conditions to assess real-world reliability alongside Bluetooth and Wi-Fi interference in the shared 2.4 GHz band.

### Why This Matters

A trained model deployed on a low-cost RF sensor gives a security team actionable, real-time alerts — *"drone detected, likely Phantom, flight mode: hovering"* — without requiring line-of-sight, daylight, or the drone's cooperation. That output is understandable and actionable by non-technical personnel.


---
## 2. Data Sources & Structure


In [1]:
# Library imports — uncomment pip installs if running in a fresh Colab environment
# !pip install scipy pandas numpy matplotlib scikit-learn -q

import os, re, sys
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import signal as scipy_signal, io as sio
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 100

print("Python:", sys.version.split()[0])
print("NumPy:", np.__version__, "| Pandas:", pd.__version__)


Python: 3.12.3
NumPy: 2.4.4 | Pandas: 3.0.2


### 2.1 DroneRF Dataset

**Source:** Allahham et al. (2019), Mendeley Data  
**Link:** https://data.mendeley.com/datasets/f4c2b4n755/1  
**Size:** ~40 GB | **Format:** CSV (one row per file = 1,000,000 amplitude samples)

The dataset contains **227 recorded segments** from three drones — Parrot Bebop, Parrot AR, and DJI Phantom — each operating in four flight modes, plus background RF recordings with no drone present. Each segment is split into two files capturing the low and high halves of the 2.4 GHz band.

**Flight modes recorded:**
| Mode | Description |
|---|---|
| Mode 1 | On and connected to controller |
| Mode 2 | Hovering (no manual input) |
| Mode 3 | Flying without video recording |
| Mode 4 | Flying with video recording |

**Filename label scheme (BUI — Binary Unique Identifier):**

Filenames encode all label information, e.g. `11010H3.csv`:

| BUI bits | Meaning |
|---|---|
| Bit 1 | `0` = background (no drone), `1` = drone present |
| Bits 2–3 | Drone type: `00`=Bebop, `01`=AR, `10`=Phantom |
| Bits 4–5 | Flight mode: `00`=Mode1, `01`=Mode2, `10`=Mode3, `11`=Mode4 |
| `L` or `H` | Low or High frequency half of the spectrum |
| Trailing number | Segment index |


---
## 3. Data Loading & Structure Exploration


In [2]:
import sys, os
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

# ── PATH SETUP ─────────────────────────────────────────────────────────────────
# DroneRF files are stored under:
#   /content/drive/MyDrive/DroneRF/
# with subfolders: AR drone, Bebop drone, Phantom drone, Background RF activites
# CSV files are extracted in-place from .rar archives (extraction cell above).
# The loader searches all subfolders recursively so no path changes are needed.

DRONERF_ROOT = "/content/drive/MyDrive/DroneRF" if IN_COLAB else "./data/DroneRF"

CONFIG = {
    "dronerf_root": DRONERF_ROOT,
    "processed_dir": "/content/drive/MyDrive/capstone_processed" if IN_COLAB else "./data/processed",
}

os.makedirs(CONFIG["processed_dir"], exist_ok=True)
print("DroneRF root :", CONFIG["dronerf_root"])
print("Processed dir:", CONFIG["processed_dir"])


DroneRF root: ./data/DroneRF
Processed output: ./data/processed


In [ ]:
# ── EXTRACT .rar FILES ─────────────────────────────────────────────────────────
# Only needs to run once. If CSVs are already extracted, this cell is safe to
# skip or re-run (the -y flag auto-confirms overwrites).

import subprocess, os
from pathlib import Path

!apt-get install -y unrar -q

rar_root = CONFIG["dronerf_root"]
rar_files = list(Path(rar_root).rglob("*.rar"))
print(f"Found {len(rar_files)} .rar files")

for rar_path in rar_files:
    result = subprocess.run(
        ["unrar", "e", "-y", str(rar_path), str(rar_path.parent) + "/"],
        capture_output=True, text=True
    )
    status = "OK" if result.returncode == 0 else f"ERROR: {result.stderr[:80]}"
    print(f"  {rar_path.name} → {status}")

# Count extracted CSVs
csv_count = len(list(Path(rar_root).rglob("*.csv")))
print(f"\nExtraction complete. Total CSV files found: {csv_count}")


In [3]:
# ── DroneRF label decoder ──────────────────────────────────────────────────────
DRONE_TYPE_MAP  = {"00": "Bebop", "01": "AR", "10": "Phantom"}
FLIGHT_MODE_MAP = {
    "00": "Mode1_On_Connected",
    "01": "Mode2_Hovering",
    "10": "Mode3_Flying",
    "11": "Mode4_Flying_Video",
}
FNAME_PATTERN = re.compile(r"^(?P<bui>[01]{5})(?P<half>[LH])_?(?P<segment>\d+)\.csv$")

def parse_bui(bui):
    presence_bit, type_bits, mode_bits = bui[0], bui[1:3], bui[3:5]
    if presence_bit == "0":
        return False, "Background", None
    return True, DRONE_TYPE_MAP.get(type_bits, "Unknown"), FLIGHT_MODE_MAP.get(mode_bits, "Unknown")

def build_dronerf_manifest(root_dir):
    rows, skipped = [], []
    for path in Path(root_dir).rglob("*.csv"):
        m = FNAME_PATTERN.match(path.name)
        if not m:
            skipped.append(path.name); continue
        bui = m.group("bui")
        drone_present, drone_type, flight_mode = parse_bui(bui)
        rows.append({
            "filepath": str(path), "bui": bui, "half": m.group("half"),
            "segment_id": int(m.group("segment")),
            "drone_present": drone_present, "drone_type": drone_type,
            "flight_mode": flight_mode,
        })
    if skipped:
        print(f"Skipped {len(skipped)} files with unexpected names (e.g. {skipped[:3]})")
    return pd.DataFrame(rows)

def pair_dronerf_segments(manifest):
    pivoted = manifest.pivot_table(
        index=["bui", "segment_id", "drone_present", "drone_type", "flight_mode"],
        columns="half", values="filepath", aggfunc="first"
    ).reset_index().rename(columns={"L": "low_filepath", "H": "high_filepath"})
    incomplete = pivoted[pivoted["low_filepath"].isna() | pivoted["high_filepath"].isna()]
    if len(incomplete):
        print(f"Warning: {len(incomplete)} incomplete segment(s) — missing one half-file")
    return pivoted

# Load
dronerf_manifest  = build_dronerf_manifest(CONFIG["dronerf_root"])
dronerf_segments  = pair_dronerf_segments(dronerf_manifest) if len(dronerf_manifest) else pd.DataFrame()
print(f"DroneRF: {len(dronerf_manifest)} half-files → {len(dronerf_segments)} paired segments")
dronerf_segments.head()


DroneRF: 0 half-files → 0 paired segments


""


---
## 4. Exploratory Data Analysis (EDA)


In [4]:
# ── Class distribution ─────────────────────────────────────────────────────────
if len(dronerf_segments):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    # Drone presence (binary task)
    presence_counts = dronerf_segments["drone_present"].value_counts()
    presence_counts.index = ["Drone Present" if v else "Background" for v in presence_counts.index]
    axes[0].bar(presence_counts.index, presence_counts.values, color=["#003262", "#FDB515"])
    axes[0].set_title("Task 1: Drone Presence (Binary)")
    axes[0].set_ylabel("Number of segments")
    for i, v in enumerate(presence_counts.values):
        axes[0].text(i, v + 0.3, str(v), ha="center", fontweight="bold")

    # Drone type + flight mode (multi-class task)
    type_counts = dronerf_segments.query("drone_present")["drone_type"].value_counts()
    axes[1].bar(type_counts.index, type_counts.values, color=["#003262", "#FDB515", "#888"])
    axes[1].set_title("Task 2: Drone Type (Multi-class)")
    axes[1].set_ylabel("Number of segments")
    for i, v in enumerate(type_counts.values):
        axes[1].text(i, v + 0.3, str(v), ha="center", fontweight="bold")

    plt.suptitle("DroneRF Dataset — Class Distribution", fontsize=13, fontweight="bold")
    plt.tight_layout()
    plt.savefig(os.path.join(CONFIG["processed_dir"], "class_distribution.png"), bbox_inches="tight")
    plt.show()
    print("\nFlight mode breakdown:")
    display(dronerf_segments.groupby(["drone_type", "flight_mode"]).size().reset_index(name="count"))
else:
    print("No DroneRF data loaded yet — set CONFIG['dronerf_root'] and re-run.")


No DroneRF data loaded yet — set CONFIG['dronerf_root'] and re-run.


In [5]:
# ── Sample waveform + spectrogram visualization ────────────────────────────────
def load_signal_csv(filepath):
    return pd.read_csv(filepath, header=None).values.flatten()

def plot_segment(low_path, high_path, label="", fs=40e6, ax_wave=None, ax_spec=None):
    """Plot waveform and STFT spectrogram for one paired segment."""
    sig = np.concatenate([load_signal_csv(low_path), load_signal_csv(high_path)])
    f, t, Sxx = scipy_signal.stft(sig, fs=fs, nperseg=512, noverlap=256)
    Sxx_db = 10 * np.log10(np.abs(Sxx) ** 2 + 1e-12)

    ax_wave.plot(sig[:4000], linewidth=0.6)
    ax_wave.set_title(f"Waveform — {label}", fontsize=9)
    ax_wave.set_xlabel("Sample index"); ax_wave.set_ylabel("Amplitude")

    ax_spec.pcolormesh(t, f / 1e6, Sxx_db, shading="auto", cmap="viridis")
    ax_spec.set_title(f"Spectrogram — {label}", fontsize=9)
    ax_spec.set_xlabel("Time [s]"); ax_spec.set_ylabel("Freq [MHz]")

if len(dronerf_segments):
    # Plot one Background and one drone segment side by side
    fig, axes = plt.subplots(2, 4, figsize=(16, 6))
    sample_pairs = [
        ("Background",       dronerf_segments.query("drone_type == 'Background'").iloc[0]),
        ("AR – Hovering",    dronerf_segments.query("drone_type == 'AR' and flight_mode == 'Mode2_Hovering'").iloc[0]),
        ("Bebop – Flying",   dronerf_segments.query("drone_type == 'Bebop' and flight_mode == 'Mode3_Flying'").iloc[0]),
        ("Phantom – Video",  dronerf_segments.query("drone_type == 'Phantom' and flight_mode == 'Mode4_Flying_Video'").iloc[0]),
    ]
    for col, (label, row) in enumerate(sample_pairs):
        plot_segment(row["low_filepath"], row["high_filepath"], label=label,
                     ax_wave=axes[0][col], ax_spec=axes[1][col])
    plt.suptitle("RF Signal Snapshots: Waveform (top) & Spectrogram (bottom)", fontsize=12, fontweight="bold")
    plt.tight_layout()
    plt.savefig(os.path.join(CONFIG["processed_dir"], "sample_spectrograms.png"), bbox_inches="tight")
    plt.show()
else:
    print("Awaiting real data — visualization will populate once DroneRF is downloaded.")


Awaiting real data — visualization will populate once DroneRF is downloaded.


---
## 5. Data Preprocessing & Preparation

### 5.1 Signal Normalization

Each segment's amplitude values are normalized to the range [−1, 1] by dividing by the maximum absolute value. This prevents segments with larger absolute amplitudes from dominating the model and is consistent with the normalization used in the original DroneRF paper.

### 5.2 STFT → Spectrogram

Raw IQ (amplitude) time-series data is converted into 2D spectrograms using Short-Time Fourier Transform (STFT). Each spectrogram image captures the **frequency content over time** — the "visual fingerprint" of a drone's RF signature that the CNN will learn to recognize.

**Parameters:**
- `nperseg = 512` — FFT window size (balances frequency resolution vs. time resolution)
- `noverlap = 256` — 50% overlap between windows
- `fs = 40e6` — sample rate (confirm against dataset documentation)
- Output is converted to dB scale: `10 * log10(|Sxx|² + ε)`

### 5.3 Handling Class Imbalance

The DroneRF dataset has more background (no-drone) recordings than individual drone-type recordings. Strategies to address this:
- **Weighted loss function** — assigns higher loss penalty to minority classes during training
- **Stratified train/val/test split** — ensures each class is proportionally represented in all splits

### 5.4 Train / Validation / Test Split

Splits are performed **by segment** (not by individual sample), so no physical recording leaks across splits:
- **70%** training | **15%** validation | **15%** test
- Stratified by `(drone_type, flight_mode)` to preserve class balance across all three sets


In [6]:
# ── STFT preprocessing pipeline ────────────────────────────────────────────────
def segment_to_spectrogram(low_path, high_path, fs=40e6, nperseg=512, noverlap=256,
                            target_shape=(128, 128)):
    """Load a paired DroneRF segment and return a normalized spectrogram array."""
    low  = load_signal_csv(low_path)
    high = load_signal_csv(high_path)
    sig  = np.concatenate([low, high])

    # Amplitude normalization
    max_amp = np.max(np.abs(sig))
    if max_amp > 0:
        sig = sig / max_amp

    _, _, Sxx = scipy_signal.stft(sig, fs=fs, nperseg=nperseg, noverlap=noverlap)
    Sxx_db = 10 * np.log10(np.abs(Sxx) ** 2 + 1e-12)

    # Resize to a fixed shape so all images are the same size for the CNN
    from scipy.ndimage import zoom
    zoom_factors = (target_shape[0] / Sxx_db.shape[0], target_shape[1] / Sxx_db.shape[1])
    Sxx_resized = zoom(Sxx_db, zoom_factors)

    # Min-max scale to [0, 1] for model input
    vmin, vmax = Sxx_resized.min(), Sxx_resized.max()
    if vmax > vmin:
        Sxx_resized = (Sxx_resized - vmin) / (vmax - vmin)

    return Sxx_resized   # shape: (128, 128)

# Test on one segment if data is loaded
if len(dronerf_segments):
    row = dronerf_segments.iloc[0]
    spec = segment_to_spectrogram(row["low_filepath"], row["high_filepath"])
    print("Spectrogram shape:", spec.shape, "| min:", spec.min().round(4), "| max:", spec.max().round(4))

    fig, ax = plt.subplots(figsize=(5, 4))
    ax.imshow(spec, aspect="auto", origin="lower", cmap="viridis")
    ax.set_title(f"Preprocessed spectrogram — {row['drone_type']} / {row['flight_mode']}")
    ax.set_xlabel("Time bins"); ax.set_ylabel("Frequency bins")
    plt.tight_layout(); plt.show()
else:
    print("Preprocessing pipeline is ready — will run once data is loaded.")


Preprocessing pipeline is ready — will run once data is loaded.


In [7]:
# ── Build dataset (X, y) from all paired segments ─────────────────────────────
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

def build_dataset(segments_df, task="presence"):
    """
    task = 'presence' : binary (drone / no drone)
    task = 'type'     : multi-class (Background, AR, Bebop, Phantom)
    task = 'mode'     : multi-class (drone-only segments, flight mode)
    """
    X, y_raw = [], []
    for _, row in segments_df.iterrows():
        try:
            spec = segment_to_spectrogram(row["low_filepath"], row["high_filepath"])
            X.append(spec)
            if task == "presence":
                y_raw.append(int(row["drone_present"]))
            elif task == "type":
                y_raw.append(row["drone_type"])
            elif task == "mode":
                if row["drone_present"]:
                    y_raw.append(row["flight_mode"])
                else:
                    continue
        except Exception as e:
            print(f"  Skipped segment {row['bui']}-{row['segment_id']}: {e}")

    X = np.array(X)[..., np.newaxis]   # add channel dim for CNN: (N, 128, 128, 1)
    le = LabelEncoder()
    y = le.fit_transform(y_raw)
    return X, y, le

if len(dronerf_segments):
    print("Building dataset for presence detection task...")
    X, y, le = build_dataset(dronerf_segments, task="presence")
    print(f"X shape: {X.shape}  |  y shape: {y.shape}")
    print("Classes:", le.classes_)

    X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.30,
                                                          stratify=y, random_state=42)
    X_val, X_test, y_val, y_test     = train_test_split(X_temp, y_temp, test_size=0.50,
                                                          stratify=y_temp, random_state=42)
    print(f"Train: {len(X_train)} | Val: {len(X_val)} | Test: {len(X_test)}")
else:
    print("Dataset builder is ready — will run once DroneRF data is loaded.")


Dataset builder is ready — will run once DroneRF data is loaded.

---
## 6. Modeling

Three modeling approaches are planned, applied sequentially to build complexity and interpretability:

| Model | Rationale |
|---|---|
| **CNN on Spectrograms** | Spectrograms are 2D images — CNNs are the natural architecture for learning spatial frequency patterns |
| **LSTM / RNN** | Captures *temporal* patterns across the signal over time — complements the CNN's spatial view |
| **Baseline: Logistic Regression / SVM on PSD features** | Provides a simple, interpretable benchmark to verify the deep learning models are adding value |


In [8]:
# ── CNN model definition ────────────────────────────────────────────────────────
# Requires TensorFlow/Keras — install if needed:
# !pip install tensorflow -q

try:
    import tensorflow as tf
    from tensorflow.keras import layers, models

    def build_cnn(input_shape=(128, 128, 1), num_classes=2):
        """Lightweight CNN for spectrogram-based drone detection/classification."""
        model = models.Sequential([
            layers.Input(shape=input_shape),

            # Block 1
            layers.Conv2D(32, (3, 3), activation="relu", padding="same"),
            layers.BatchNormalization(),
            layers.MaxPooling2D((2, 2)),
            layers.Dropout(0.25),

            # Block 2
            layers.Conv2D(64, (3, 3), activation="relu", padding="same"),
            layers.BatchNormalization(),
            layers.MaxPooling2D((2, 2)),
            layers.Dropout(0.25),

            # Block 3
            layers.Conv2D(128, (3, 3), activation="relu", padding="same"),
            layers.BatchNormalization(),
            layers.MaxPooling2D((2, 2)),
            layers.Dropout(0.25),

            # Classifier head
            layers.Flatten(),
            layers.Dense(256, activation="relu"),
            layers.Dropout(0.5),
            layers.Dense(num_classes, activation="softmax" if num_classes > 2 else "sigmoid"),
        ])

        model.compile(
            optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
            loss="sparse_categorical_crossentropy" if num_classes > 2 else "binary_crossentropy",
            metrics=["accuracy"],
        )
        return model

    model = build_cnn(input_shape=(128, 128, 1), num_classes=2)  # binary: drone / no drone
    model.summary()

except ImportError:
    print("TensorFlow not available in this environment.")
    print("Install with: pip install tensorflow")
    print("Model architecture is defined and ready — install TF and re-run to train.")


TensorFlow not available in this environment.
Install with: pip install tensorflow
Model architecture is defined and ready — install TF and re-run to train.


In [9]:
# ── Training ────────────────────────────────────────────────────────────────────
# NOTE: This cell requires real DroneRF data to have been loaded above.
# Training on the full dataset will take significant time on CPU;
# use a GPU runtime in Colab (Runtime > Change runtime type > T4 GPU).

if len(dronerf_segments) and "X_train" in dir():
    from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

    callbacks = [
        EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True, verbose=1),
        ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3, verbose=1),
    ]

    history = model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=30,
        batch_size=16,
        callbacks=callbacks,
        verbose=1,
    )

    # Save model
    model.save(os.path.join(CONFIG["processed_dir"], "cnn_drone_detector.keras"))
    print("Model saved.")
else:
    print("Training will run once real data is loaded and pre-processed.")
    print("Expected training time on Colab GPU (T4): ~15–30 min for 30 epochs on DroneRF.")


Training will run once real data is loaded and pre-processed.


Expected training time on Colab GPU (T4): ~15–30 min for 30 epochs on DroneRF.


---
## 7. Model Evaluation

### Evaluation Metrics

For the **binary task** (drone present/absent):
- Accuracy, Precision, Recall, F1-Score
- ROC-AUC curve

For the **multi-class task** (drone type + flight mode):
- Per-class F1-Score
- Confusion matrix (normalized)
- Macro and weighted F1 averages

### SNR Robustness Testing

A key evaluation dimension is how performance degrades under increasing noise — specifically because Bluetooth and Wi-Fi devices share the 2.4 GHz band with consumer drones. Test subsets will be grouped by estimated SNR level to produce a performance-vs-SNR curve.


In [10]:
# ── Evaluation on held-out test set ────────────────────────────────────────────
from sklearn.metrics import (classification_report, confusion_matrix,
                              ConfusionMatrixDisplay, roc_curve, auc)

def evaluate_model(model, X_test, y_test, label_encoder, task_name=""):
    y_pred_prob = model.predict(X_test, verbose=0)
    if y_pred_prob.shape[1] > 1:
        y_pred = np.argmax(y_pred_prob, axis=1)
    else:
        y_pred = (y_pred_prob[:, 0] > 0.5).astype(int)

    print(f"\n=== {task_name} — Classification Report ===")
    print(classification_report(y_test, y_pred, target_names=label_encoder.classes_))

    # Confusion matrix
    fig, ax = plt.subplots(figsize=(6, 5))
    cm = confusion_matrix(y_test, y_pred, normalize="true")
    disp = ConfusionMatrixDisplay(cm, display_labels=label_encoder.classes_)
    disp.plot(ax=ax, colorbar=False, cmap="Blues")
    ax.set_title(f"Confusion Matrix — {task_name}")
    plt.tight_layout()
    plt.savefig(os.path.join(CONFIG["processed_dir"], f"confusion_{task_name.replace(' ','_')}.png"),
                bbox_inches="tight")
    plt.show()
    return y_pred

# ── Training history plot ───────────────────────────────────────────────────────
def plot_training_history(history):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(history.history["loss"], label="Train loss")
    axes[0].plot(history.history["val_loss"], label="Val loss")
    axes[0].set_title("Loss"); axes[0].legend()

    axes[1].plot(history.history["accuracy"], label="Train acc")
    axes[1].plot(history.history["val_accuracy"], label="Val acc")
    axes[1].set_title("Accuracy"); axes[1].legend()
    plt.suptitle("Training History", fontweight="bold")
    plt.tight_layout()
    plt.savefig(os.path.join(CONFIG["processed_dir"], "training_history.png"), bbox_inches="tight")
    plt.show()

if "history" in dir():
    plot_training_history(history)
    y_pred = evaluate_model(model, X_test, y_test, le, task_name="Drone Presence Detection")
else:
    print("Evaluation cells ready — will populate once model is trained.")


Evaluation cells ready — will populate once model is trained.


---
## 8. Results Summary

> **Note:** This section will be updated with final metrics after model training completes.  
> Results below are placeholder targets based on published benchmarks for similar approaches.

| Task | Model | Expected Accuracy | Status |
|---|---|---|---|
| Drone presence detection | CNN (spectrograms) | >95% | ⏳ Pending training |
| Drone type classification | CNN (spectrograms) | >90% | ⏳ Pending training |
| Flight mode classification | CNN (spectrograms) | >85% | ⏳ Pending training |
| Drone presence detection | LSTM | TBD | ⏳ Pending |
| Drone presence detection | Logistic Regression (baseline) | ~80% | ⏳ Pending |

---
## 9. Next Steps

- [ ] Confirm BUI label scheme against the DroneRF dataset documentation PDF
- [ ] Complete spectrogram preprocessing over all segments; save to processed_dir for faster re-loading
- [ ] Train and evaluate CNN (binary detection task first, then multi-class)
- [ ] Build LSTM model and compare against CNN
- [ ] Run SNR robustness evaluation (vary noise level; plot performance vs. SNR)
- [ ] Write final non-technical report (Module 24)

---
## References

1. Allahham, M.S. et al. (2019). *DroneRF dataset: A dataset of drones for RF-based detection, classification, and identification.* Data in Brief, 26, 104313.
2. Al-Sa'd, M. et al. (2019). *RF-based drone detection and identification using deep learning approaches.* Future Generation Computer Systems, 100, 86–97.
